In [1]:
import pandas as pd
import numpy as np
import random
import pickle as pkl

In [2]:
def stat_genes_filt(gene_list, gene1_list, gene2_list):

    gene1_bool = [True if g in gene_list else False for g in gene1_list]
    gene2_bool = [True if g in gene_list else False for g in gene2_list]

    total_bool = np.logical_and(gene1_bool, gene2_bool)

    print("valid data size:", np.sum(total_bool),
          "/", len(gene1_bool))

    return total_bool

In [3]:
def normalize(data):

    min_x = min(data)
    max_x = max(data)
    res = [(x - min_x) / (max_x - min_x) for x in data]

    return res

In [4]:
with open("../data/saved_data/map/gene_list.txt") as f:
        gene_list = [line.rstrip('\n') for line in f]

with open("../data/saved_data/map/gene2id.pkl", 'rb') as f:
    gene2id_map =  pkl.load(f)

id2gene_map = {i:g for g,i in gene2id_map.items()}

#### DBKO

Study1, PMID=28319113

In [5]:
df1 = pd.read_excel("../data/SL_data/raw/SLlabel_DBKO.xlsx", sheet_name=0, skiprows=2)
df1.head()

,geneA,geneB,Hit_Cell_Line,Interaction_type
0,BRCA1,WEE1,293T,Synthetic Lethal
1,BRCA2,CDK9,293T,Synthetic Lethal
2,FNTA,TOP1,293T,Synthetic Lethal
3,CHEK1,IGF1R,293T,Synthetic Lethal
4,CHEK1,VEGFA,293T,Synthetic Lethal


In [6]:
df1_cellline_context_map = {
    "A549": "LUAD",
    "Hela": "CESC",
    "HeLa": "CESC"
}

In [7]:
df1_new = []

for idx, row in df1.iterrows():

    c_split = row["Hit_Cell_Line"].split(",")
    for c in c_split:
        if c in df1_cellline_context_map.keys():
            df1_new.append({
                "gene1": row["geneA"],
                "gene2": row["geneB"],
                "cancer": df1_cellline_context_map[c],
                "label": 1
            })

In [8]:
df1_new_pd = pd.DataFrame(df1_new)
df1_new_pd

,gene1,gene2,cancer,label
0,IGF1R,PRKDC,LUAD,1
1,PRKDC,RRM2,LUAD,1
2,CHEK1,SMO,CESC,1
3,ERBB2,TOP1,CESC,1
4,CHEK1,PRKDC,CESC,1
...,...,...,...,...
92,ERBB2,SRC,LUAD,1
93,BRAF,EGFR,LUAD,1
94,CHEK2,KMT2D,CESC,1
95,MAP2K1,VHL,CESC,1


In [9]:
df1_new_pd_LUAD = df1_new_pd[df1_new_pd['cancer']=="LUAD"]
df1_new_pd_CESC = df1_new_pd[df1_new_pd['cancer']=="CESC"]

In [55]:
df1_new_pd_LUAD.to_csv("../data/SL_data/processed/SLlabel_DBKO_study1_LUAD.csv", index=False)
df1_new_pd_CESC.to_csv("../data/SL_data/processed/SLlabel_DBKO_study1_CESC.csv", index=False)

Study2, PMID=30033366

In [11]:
df2 = pd.read_excel("../data/SL_data/raw/SLlabel_DBKO.xlsx", sheet_name=1, skiprows=3)
df2['K562_ravg_normalized'] = normalize(df2['K562_ravg'])
df2.head()

,gene_A,gene_B,Jurkat_ravg,K562_ravg,K562_ravg_normalized
0,AARS2,AATF,-0.19968,-0.54548,0.568926
1,AARS2,ABCB7,-0.42328,-0.62515,0.490467
2,AARS2,ACTL6A,-0.29358,-0.71750,0.399521
3,AARS2,ACTR10,-0.18909,-0.67805,0.438372
4,AARS2,ADAT2,-0.17961,-0.57520,0.539658


In [12]:
df2_new = []

for idx, row in df2.iterrows():
    if row["gene_A"] != "negative" and row["gene_B"] != "negative":
        df2_new.append({
                "gene1": row["gene_A"],
                "gene2": row["gene_B"],
                "cancer": "LAML",
                "GI": row["K562_ravg_normalized"]
        })

In [13]:
df2_new_pd = pd.DataFrame(df2_new)
df2_new_pd.sort_values(by='GI', inplace=True)
df2_new_downsampled = pd.concat([df2_new_pd.iloc[:250,:], df2_new_pd.iloc[-250:,:]])
df2_new_downsampled

,gene1,gene2,cancer,GI
49224,MED23,RPL23,LAML,0.000000
74331,NUP54,MED22,LAML,0.014723
3343,ARGLU1,PELO,LAML,0.016870
110620,SPAG7,CDC16,LAML,0.054656
83118,POLR2K,C11orf30,LAML,0.062072
...,...,...,...,...
99022,RPL35,EXOSC4,LAML,0.948308
112996,SYVN1,XRN2,LAML,0.954887
76224,PAFAH1B1,WARS2,LAML,0.958836
49767,MED28,WARS2,LAML,0.982461


In [18]:
df2_new_downsampled.to_csv("../data/SL_data/processed/SLlabel_DBKO_study2.csv", index=False)

Study3, PMID=28319085

In [15]:
df3 = pd.read_excel("../data/SL_data/raw/SLlabel_DBKO.xlsx", sheet_name=3, skiprows=2)
df3["GIT_score_normalized"] = normalize(df3["GeneAB GIT score"])
df3.head()

,Gene_Pair,GeneAB GIT score,GIT_score_normalized
0,AKT1__AKT2,-10.003,0.000000
1,PIM1__PIM2,-9.856,0.007772
2,BCL2L1__MCL1,-8.359,0.086924
3,TK1__TK1,-8.116,0.099773
4,ALAD__GPI,-7.048,0.156242


In [16]:
genea_3 = [comb.split("__")[0] for comb in df3.iloc[:, 0]]
geneb_3 = [comb.split("__")[1] for comb in df3.iloc[:, 0]]
print(genea_3[:5], geneb_3[:5])

['AKT1', 'PIM1', 'BCL2L1', 'TK1', 'ALAD'] ['AKT2', 'PIM2', 'MCL1', 'TK1', 'GPI']


In [17]:
df3.insert(loc=0, column="geneA", value=genea_3)
df3.insert(loc=1, column="geneB", value=geneb_3)
df3.head()

,geneA,geneB,Gene_Pair,GeneAB GIT score,GIT_score_normalized
0,AKT1,AKT2,AKT1__AKT2,-10.003,0.000000
1,PIM1,PIM2,PIM1__PIM2,-9.856,0.007772
2,BCL2L1,MCL1,BCL2L1__MCL1,-8.359,0.086924
3,TK1,TK1,TK1__TK1,-8.116,0.099773
4,ALAD,GPI,ALAD__GPI,-7.048,0.156242


In [18]:
df3_new = []

for idx, row in df3.iterrows():
    df3_new.append({
            "gene1": row["geneA"],
            "gene2": row["geneB"],
            "cancer": "LAML",
            "GI": row["GIT_score_normalized"]
    })

In [19]:
df3_new_pd = pd.DataFrame(df3_new)
df3_new_pd

,gene1,gene2,cancer,GI
0,AKT1,AKT2,LAML,0.000000
1,PIM1,PIM2,LAML,0.007772
2,BCL2L1,MCL1,LAML,0.086924
3,TK1,TK1,LAML,0.099773
4,ALAD,GPI,LAML,0.156242
...,...,...,...,...
20985,PIK3CA,PIK3CA,LAML,0.917411
20986,LCK,LCK,LAML,0.961508
20987,NPY4R,NPY4R,LAML,0.977106
20988,AKT2,AKT2,LAML,0.991170


In [20]:
df3_new_downsampled = pd.concat([df3_new_pd.iloc[:250,:], df3_new_pd.iloc[-250:,:]])
df3_new_downsampled

,gene1,gene2,cancer,GI
0,AKT1,AKT2,LAML,0.000000
1,PIM1,PIM2,LAML,0.007772
2,BCL2L1,MCL1,LAML,0.086924
3,TK1,TK1,LAML,0.099773
4,ALAD,GPI,LAML,0.156242
...,...,...,...,...
20985,PIK3CA,PIK3CA,LAML,0.917411
20986,LCK,LCK,LAML,0.961508
20987,NPY4R,NPY4R,LAML,0.977106
20988,AKT2,AKT2,LAML,0.991170


In [26]:
df3_new_downsampled.to_csv("../data/SL_data/processed/SLlabel_DBKO_study3.csv", index=False)

#### Prim partner

Study1, PMID=19490893

In [21]:
df1 = pd.read_excel("../data/SL_data/raw/SLlabel_primpartner.xlsx", sheet_name=0, skiprows=4)
df1.columns = ["partner_gene", "DLD-1_fitness_mean", "DLD-1_fitness_sd", "HCT116_fitness_mean", "HCT116_fitness_sd"]
df1.insert(loc=0, column="primary_gene", value=["KRAS"]*len(df1))
df1.head()

,primary_gene,partner_gene,DLD-1_fitness_mean,DLD-1_fitness_sd,HCT116_fitness_mean,HCT116_fitness_sd
0,KRAS,ANAPC1,82.2,3.2,48.3,3.7
1,KRAS,ANAPC1,55.1,1.9,81.0,6.9
2,KRAS,ANAPC1,75.5,1.7,104.9,10.7
3,KRAS,ANAPC4,92.1,5.8,51.4,1.7
4,KRAS,ANAPC4,84.8,4.4,59.2,4.8


In [22]:
# drop duplicates
print(len(df1))
df1_filt = df1.drop_duplicates(subset=['primary_gene', 'partner_gene'], keep='first')
print(len(df1_filt))

83
77


In [23]:
df1_new_DLD1 = []
df1_new_HCT116 = []

for idx, row in df1_filt.iterrows():
    
    df1_new_DLD1.append({
        "gene1": row["primary_gene"],
        "gene2": row["partner_gene"],
        "cancer": "COAD",
        "fitness": row["DLD-1_fitness_mean"]
    })

    df1_new_HCT116.append({
        "gene1": row["primary_gene"],
        "gene2": row["partner_gene"],
        "cancer": "COAD",
        "fitness": row["HCT116_fitness_mean"]
    })

In [24]:
df1_new_DLD1_pd = pd.DataFrame(df1_new_DLD1)
df1_new_HCT116_pd = pd.DataFrame(df1_new_HCT116)
df1_new_DLD1_pd

,gene1,gene2,cancer,fitness
0,KRAS,ANAPC1,COAD,82.2
1,KRAS,ANAPC4,COAD,92.1
2,KRAS,BXDC1,COAD,84.6
3,KRAS,BXDC2,COAD,79.4
4,KRAS,BXDC5,COAD,71.4
...,...,...,...,...
72,KRAS,USP39,COAD,58.4
73,KRAS,WDSOF1,COAD,82.1
74,KRAS,XPO1,COAD,78.6
75,KRAS,ZC3H18,COAD,80.7


In [25]:
df1_new_DLD1_pd = df1_new_DLD1_pd.dropna(how = 'any')
df1_new_HCT116_pd = df1_new_HCT116_pd.dropna(how = 'any')
print(len(df1_new_DLD1_pd), len(df1_new_HCT116_pd))

77 62


In [9]:
df1_new_DLD1_pd.to_csv("../data/SL_data/processed/SLlabel_primpartner_study1_DLD1.csv", index=False)
df1_new_HCT116_pd.to_csv("../data/SL_data/processed/SLlabel_primpartner_study1_HCT116.csv", index=False)


In [26]:
# 0/1 labels

random.seed(1)
sampling_set = list(set(gene_list).difference(set(list(df1_filt["partner_gene"])+["KRAS"])))
random.shuffle(sampling_set)
neg_sample = sampling_set[:len(df1_filt)]


In [27]:
# 0/1 labels

df1_new_binary = []

for idx, row in df1_filt.iterrows():
    
    df1_new_binary.append({
        "gene1": row["primary_gene"],
        "gene2": row["partner_gene"],
        "cancer": "COAD",
        "label": 1
    })

for g in neg_sample:
    df1_new_binary.append({
        "gene1": row["primary_gene"],
        "gene2": g,
        "cancer": "COAD",
        "label": 0
    })

In [28]:
df1_new_binary_pd = pd.DataFrame(df1_new_binary)
df1_new_binary_pd

,gene1,gene2,cancer,label
0,KRAS,ANAPC1,COAD,1
1,KRAS,ANAPC4,COAD,1
2,KRAS,BXDC1,COAD,1
3,KRAS,BXDC2,COAD,1
4,KRAS,BXDC5,COAD,1
...,...,...,...,...
149,KRAS,CRABP2,COAD,0
150,KRAS,NAGK,COAD,0
151,KRAS,SDC4,COAD,0
152,KRAS,CALR,COAD,0


In [29]:
df1_new_binary_pd.to_csv("../data/SL_data/processed/SLlabel_primpartner_study1_binary.csv", index=False)

*Study2, PMID=28102363 (tagal 2017)

In [30]:
df2 = pd.read_excel("../data/SL_data/raw/SLlabel_primpartner.xlsx", sheet_name=1, skiprows=3)
df2.insert(loc=0, column="primary_gene", value=["SMARCA4"]*len(df2))
df2["Mean_Viability_normalized"] = normalize(df2["Mean Viability"])
df2.head()

,primary_gene,Gene,Viability-Well #1,Viability-Well #2,Viability-Well #3,Mean Viability,Z Score,Robust Z Score,Mean_Viability_normalized
0,SMARCA4,UBB,0.027967,0.029820,0.030972,0.03,-12.172,-29.941,0.000000
1,SMARCA4,UBC,0.031138,0.029699,0.025760,0.03,-12.073,-29.700,0.000000
2,SMARCA4,NXF1,0.173168,0.170848,0.169705,0.17,-10.446,-15.725,0.075269
3,SMARCA4,TPX2,0.258938,0.221477,0.227850,0.22,-9.314,-14.770,0.102151
4,SMARCA4,POLR2A,0.289478,0.296489,0.294609,0.29,-4.156,-10.423,0.139785


In [31]:
df2_new = []

for idx, row in df2.iterrows():
    
    df2_new.append({
        "gene1": row["primary_gene"],
        "gene2": row["Gene"],
        "cancer": "LUAD",
        "viability": row["Mean_Viability_normalized"]
    })

In [32]:
df2_new_pd = pd.DataFrame(df2_new)
df2_new_pd.head()

,gene1,gene2,cancer,viability
0,SMARCA4,UBB,LUAD,0.000000
1,SMARCA4,UBC,LUAD,0.000000
2,SMARCA4,NXF1,LUAD,0.075269
3,SMARCA4,TPX2,LUAD,0.102151
4,SMARCA4,POLR2A,LUAD,0.139785


In [33]:
len(df2_new_pd)

21124

In [34]:
df2_new_downsampled = pd.concat([df2_new_pd.iloc[:250,:], df2_new_pd.iloc[-250:,:]])
df2_new_downsampled

,gene1,gene2,cancer,viability
0,SMARCA4,UBB,LUAD,0.000000
1,SMARCA4,UBC,LUAD,0.000000
2,SMARCA4,NXF1,LUAD,0.075269
3,SMARCA4,TPX2,LUAD,0.102151
4,SMARCA4,POLR2A,LUAD,0.139785
...,...,...,...,...
21119,SMARCA4,ATXN7,LUAD,0.860215
21120,SMARCA4,RPL22,LUAD,0.887097
21121,SMARCA4,UBL4A,LUAD,0.903226
21122,SMARCA4,SPTB,LUAD,0.908602


In [34]:
df2_new_downsampled.to_csv("../data/SL_data/processed/SLlabel_primpartner_study2_viability.csv", index=False)

In [35]:
# binary

df2_pos = df2[df2["Mean Viability"] <= 0.5]
len(df2_pos)

46

In [36]:
df2.sort_values(by='Mean Viability', inplace=True)
filt_bool = stat_genes_filt(gene_list, df2["primary_gene"], df2["Gene"])
df2_filt = df2[filt_bool]
len(df2_filt)

valid data size: 12070 / 21124


12070

In [37]:
df2_neg = df2_filt.iloc[-len(df2_pos):, :]
df2_neg.head()

,primary_gene,Gene,Viability-Well #1,Viability-Well #2,Viability-Well #3,Mean Viability,Z Score,Robust Z Score,Mean_Viability_normalized
21050,SMARCA4,CCDC14,1.442284,1.358689,1.424413,1.41,2.100,2.693,0.741935
21053,SMARCA4,ANKRD36,1.403486,1.371961,1.459644,1.41,2.240,2.917,0.741935
21062,SMARCA4,ANKS3,1.451176,1.439005,1.375065,1.42,2.304,2.848,0.747312
21058,SMARCA4,KIF2A,1.384693,1.786842,1.446006,1.42,1.936,2.642,0.747312
21059,SMARCA4,RPP14,1.360155,1.352899,1.545388,1.42,2.971,3.945,0.747312


In [38]:
df2_binary = []

for idx, row in df2_pos.iterrows():
    
    df2_binary.append({
        "gene1": row["primary_gene"],
        "gene2": row["Gene"],
        "cancer": "LUAD",
        "label": 1
    })

for idx, row in df2_neg.iterrows():
    
    df2_binary.append({
        "gene1": row["primary_gene"],
        "gene2": row["Gene"],
        "cancer": "LUAD",
        "label": 0
    })

In [39]:
df2_binary_pd = pd.DataFrame(df2_binary)
df2_binary_pd.head()

,gene1,gene2,cancer,label
0,SMARCA4,UBB,LUAD,1
1,SMARCA4,UBC,LUAD,1
2,SMARCA4,NXF1,LUAD,1
3,SMARCA4,TPX2,LUAD,1
4,SMARCA4,POLR2A,LUAD,1


In [24]:
df2_binary_pd.to_csv("../data/SL_data/processed/SLlabel_primpartner_study2_binary.csv", index=False)

*Study3, PMID=30686591 (mengwasser 2019)

In [40]:
df3 = pd.read_excel("../data/SL_data/raw/SLlabel_primpartner.xlsx", sheet_name=2, skiprows=4)
df3.columns = ["partner_gene", "edgeR_primary_shRNA", "edgeR_secondary_shRNA", "edgeR_colonic_CRISPR", "edgeR_ovarian_CRISPR",
               "MAGeCK_primary_shRNA", "MAGeCK_secondary_shRNA", "MAGeCK_colonic_CRISPR", "MAGeCK_ovarian_CRISPR",
               "Overall_edgeR_rank", "Overall_MAGeCK_rank", "Overall_rank"]
df3.insert(loc=0, column="primary_gene", value=["BRCA2"]*len(df3))
df3.head()

,primary_gene,partner_gene,edgeR_primary_shRNA,edgeR_secondary_shRNA,edgeR_colonic_CRISPR,edgeR_ovarian_CRISPR,MAGeCK_primary_shRNA,MAGeCK_secondary_shRNA,MAGeCK_colonic_CRISPR,MAGeCK_ovarian_CRISPR,Overall_edgeR_rank,Overall_MAGeCK_rank,Overall_rank
0,BRCA2,FEN1,309,NIL,2,3,177,NIL,2,2,1,2,1
1,BRCA2,POLQ,1,1,7,2,1,2,6,6,2,3,2
2,BRCA2,XRCC1,71,13,6,4,303,4,17,4,3,4,3
3,BRCA2,APEX2,265,5,11,15,330,5,8,8,5,5,4
4,BRCA2,UBE2A,52,NIL,1,18,65,NIL,14,19,4,7,5


In [41]:
df3_new_COAD = []
df3_new_OV = []

for idx, row in df3.iterrows():
    
    df3_new_COAD.append({
        "gene1": row["primary_gene"],
        "gene2": row["partner_gene"],
        "cancer": "COAD",
        "rank": (row["edgeR_colonic_CRISPR"]+row["MAGeCK_colonic_CRISPR"])/2
    })

    df3_new_OV.append({
        "gene1": row["primary_gene"],
        "gene2": row["partner_gene"],
        "cancer": "OV",
        "rank": (row["edgeR_ovarian_CRISPR"]+row["MAGeCK_ovarian_CRISPR"])/2
    })

In [42]:
df3_new_COAD_pd = pd.DataFrame(df3_new_COAD)
df3_new_OV_pd = pd.DataFrame(df3_new_OV)
df3_new_COAD_pd.head()

,gene1,gene2,cancer,rank
0,BRCA2,FEN1,COAD,2.0
1,BRCA2,POLQ,COAD,6.5
2,BRCA2,XRCC1,COAD,11.5
3,BRCA2,APEX2,COAD,9.5
4,BRCA2,UBE2A,COAD,7.5


In [28]:
df3_new_COAD_pd.to_csv("../data/SL_data/processed/SLlabel_primpartner_study3_COAD.csv", index=False)
df3_new_OV_pd.to_csv("../data/SL_data/processed/SLlabel_primpartner_study3_OV.csv", index=False)

#### Paralogs

*Study1, PMID=34469736 (parrish 2021)

In [44]:
df1 = pd.read_excel("../data/SL_data/raw/SLlabel_paralog.xlsx", sheet_name=0, skiprows=2)
df1['PC9_GI_score_normalized'] = normalize(df1['PC9_GI_score'])
df1['HeLa_GI_score_normalized'] = normalize(df1['HeLa_GI_score'])
df1.head()

,gene_A,gene_B,GI_flag,PC9_GI_flag,PC9_GI_score_rank,HeLa_GI_flag,HeLa_GI_score_rank,PC9_GI_score,HeLa_GI_score,PC9_GI_score_normalized,HeLa_GI_score_normalized
0,CCNL2,CCNL1,synthetic_lethal,SL_in_PC9,1,SL_in_HeLa,10,-2.313788,-1.261084,0.000000,0.299338
1,CDK6,CDK4,synthetic_lethal,SL_in_PC9,2,neither_in_HeLa,210,-1.537568,-0.251834,0.239196,0.676033
2,GSK3B,GSK3A,synthetic_lethal,SL_in_PC9,3,SL_in_HeLa,4,-1.429767,-1.659580,0.272415,0.150603
3,G3BP2,G3BP1,synthetic_lethal,SL_in_PC9,4,neither_in_HeLa,663,-1.429397,0.021784,0.272529,0.778159
4,CNOT8,CNOT7,synthetic_lethal,SL_in_PC9,5,SL_in_HeLa,1,-1.208182,-2.063079,0.340698,0.000000


In [45]:
df1_new_LUAD = []
df1_new_CESC = []

for idx, row in df1.iterrows():

    df1_new_LUAD.append({
        "gene1": row["gene_A"],
        "gene2": row["gene_B"],
        "cancer": "LUAD",
        "score": row["PC9_GI_score_normalized"]
    })

    df1_new_CESC.append({
        "gene1": row["gene_A"],
        "gene2": row["gene_B"],
        "cancer": "CESC",
        "score": row["HeLa_GI_score_normalized"]
    })

In [46]:
df1_new_LUAD_pd = pd.DataFrame(df1_new_LUAD)
df1_new_CESC_pd = pd.DataFrame(df1_new_CESC)
df1_new_LUAD_pd.head()

,gene1,gene2,cancer,score
0,CCNL2,CCNL1,LUAD,0.000000
1,CDK6,CDK4,LUAD,0.239196
2,GSK3B,GSK3A,LUAD,0.272415
3,G3BP2,G3BP1,LUAD,0.272529
4,CNOT8,CNOT7,LUAD,0.340698


In [42]:
df1_new_LUAD_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study1_score_LUAD.csv", index=False)
df1_new_CESC_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study1_score_CESC.csv", index=False)

In [47]:
df1_new_total_pd = pd.concat([df1_new_LUAD_pd, df1_new_CESC_pd])
df1_new_total_pd.head()

,gene1,gene2,cancer,score
0,CCNL2,CCNL1,LUAD,0.000000
1,CDK6,CDK4,LUAD,0.239196
2,GSK3B,GSK3A,LUAD,0.272415
3,G3BP2,G3BP1,LUAD,0.272529
4,CNOT8,CNOT7,LUAD,0.340698


In [48]:
len(df1_new_total_pd)

2060

In [44]:
df1_new_total_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study1_score.csv", index=False)

In [24]:
df1_new_binary_LUAD = []
df1_new_binary_CESC = []

for idx, row in df1.iterrows():

    if row["PC9_GI_flag"] == "SL_in_PC9":
        df1_new_binary_LUAD.append({
            "gene1": row["gene_A"],
            "gene2": row["gene_B"],
            "cancer": "LUAD",
            "class": 1
        })
    else:
        df1_new_binary_LUAD.append({
            "gene1": row["gene_A"],
            "gene2": row["gene_B"],
            "cancer": "LUAD",
            "class": 0
        })
    
    if row["HeLa_GI_flag"] == "SL_in_HeLa":
        df1_new_binary_CESC.append({
            "gene1": row["gene_A"],
            "gene2": row["gene_B"],
            "cancer": "CESC",
            "class": 1
        })
    else:
        df1_new_binary_CESC.append({
            "gene1": row["gene_A"],
            "gene2": row["gene_B"],
            "cancer": "CESC",
            "class": 0
        })

In [25]:
df1_new_binary_LUAD_pd = pd.DataFrame(df1_new_binary_LUAD)
df1_new_binary_CESC_pd = pd.DataFrame(df1_new_binary_CESC)
df1_new_binary_LUAD_pd.head()

,gene1,gene2,cancer,class
0,CCNL2,CCNL1,LUAD,1
1,CDK6,CDK4,LUAD,1
2,GSK3B,GSK3A,LUAD,1
3,G3BP2,G3BP1,LUAD,1
4,CNOT8,CNOT7,LUAD,1


In [48]:
df1_new_binary_LUAD_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study1_binary_LUAD.csv", index=False)
df1_new_binary_CESC_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study1_binary_CESC.csv", index=False)

In [26]:
df1_new_binary_total_pd = pd.concat([df1_new_binary_LUAD_pd, df1_new_binary_CESC_pd])
df1_new_binary_total_pd.head()

,gene1,gene2,cancer,class
0,CCNL2,CCNL1,LUAD,1
1,CDK6,CDK4,LUAD,1
2,GSK3B,GSK3A,LUAD,1
3,G3BP2,G3BP1,LUAD,1
4,CNOT8,CNOT7,LUAD,1


In [27]:
df1_new_binary_total_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study1_binary.csv", index=False)

*Study2, PMID=33059726 (Dede 2020)

In [48]:
df2 = pd.read_excel("../data/SL_data/raw/SLlabel_paralog.xlsx", sheet_name=1, skiprows=2)
df2['A549_normalized'] = normalize(df2['A549'])
df2['HT29_normalized'] = normalize(df2['HT29'])
df2['OVCAR8_normalized'] = normalize(df2['OVCAR8'])
df2.head()

,gene_A,gene_B,A549,HT29,OVCAR8,A549_normalized,HT29_normalized,OVCAR8_normalized
0,AP1B1,AP2B1,-1.137,-3.834,-2.483,0.525676,0.377654,0.440698
1,ARFGEF1,ARFGEF2,-2.344,-3.327,-3.914,0.406478,0.420536,0.330272
2,ATP6V0A1,ATP6V0A2,-2.583,-4.256,0.416,0.382876,0.341961,0.664403
3,CAPZA1,CAPZA2,-4.236,-4.711,-4.054,0.219633,0.303476,0.319469
4,CCNE1,CCNE2,-1.045,-2.564,-4.088,0.534762,0.485071,0.316845


In [49]:
df2_new_score_LUAD = []
df2_new_score_COAD = []
df2_new_score_OV = []

for idx, row in df2.iterrows():

    df2_new_score_LUAD.append({
        "gene1": row["gene_A"],
        "gene2": row["gene_B"],
        "cancer": "LUAD",
        "score": row["A549_normalized"]
    })

    df2_new_score_COAD.append({
        "gene1": row["gene_A"],
        "gene2": row["gene_B"],
        "cancer": "COAD",
        "score": row["HT29_normalized"]
    })

    df2_new_score_OV.append({
        "gene1": row["gene_A"],
        "gene2": row["gene_B"],
        "cancer": "OV",
        "score": row["OVCAR8_normalized"]
    })

In [50]:
df2_new_score_LUAD_pd = pd.DataFrame(df2_new_score_LUAD)
df2_new_score_COAD_pd = pd.DataFrame(df2_new_score_COAD)
df2_new_score_OV_pd = pd.DataFrame(df2_new_score_OV)
df2_new_score_OV_pd.head()

,gene1,gene2,cancer,score
0,AP1B1,AP2B1,OV,0.440698
1,ARFGEF1,ARFGEF2,OV,0.330272
2,ATP6V0A1,ATP6V0A2,OV,0.664403
3,CAPZA1,CAPZA2,OV,0.319469
4,CCNE1,CCNE2,OV,0.316845


In [51]:
df2_new_score_LUAD_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study2_score_LUAD.csv", index=False)
df2_new_score_COAD_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study2_score_COAD.csv", index=False)
df2_new_score_OV_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study2_score_OV.csv", index=False)

In [52]:
df2_new_score_total_pd = pd.concat([df2_new_score_LUAD_pd, df2_new_score_COAD_pd, df2_new_score_OV_pd])
df2_new_score_total_pd.head()

,gene1,gene2,cancer,score
0,AP1B1,AP2B1,LUAD,0.525676
1,ARFGEF1,ARFGEF2,LUAD,0.406478
2,ATP6V0A1,ATP6V0A2,LUAD,0.382876
3,CAPZA1,CAPZA2,LUAD,0.219633
4,CCNE1,CCNE2,LUAD,0.534762


In [53]:
df2_new_score_total_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study2_score.csv", index=False)

In [44]:
df2_pos = df2.iloc[:24, :]
df2_neg = df2.iloc[24:, :]

In [45]:
df2_new_binary = []

for idx, row in df2_pos.iterrows():

    for cancer_type in ["LUAD", "COAD", "OV"]:

        df2_new_binary.append({
            "gene1": row["gene_A"],
            "gene2": row["gene_B"],
            "cancer": cancer_type,
            "class": 1
        })

for idx, row in df2_neg.iterrows():

    for cancer_type in ["LUAD", "COAD", "OV"]:

        df2_new_binary.append({
            "gene1": row["gene_A"],
            "gene2": row["gene_B"],
            "cancer": cancer_type,
            "class": 0
        })

In [46]:
df2_new_binary_pd = pd.DataFrame(df2_new_binary)
df2_new_binary_pd.head()

,gene1,gene2,cancer,class
0,AP1B1,AP2B1,LUAD,1
1,AP1B1,AP2B1,COAD,1
2,AP1B1,AP2B1,OV,1
3,ARFGEF1,ARFGEF2,LUAD,1
4,ARFGEF1,ARFGEF2,COAD,1


In [47]:
df2_new_binary_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study2_binary.csv", index=False)

Study3, PMID=33637726

In [54]:
df3 = pd.read_excel("../data/SL_data/raw/SLlabel_paralog.xlsx", sheet_name=2, skiprows=2)
df3.head()

,gene_A,gene_B,cell_line,cancer
0,AP2A2,AP2A1,A375,SKCM
1,ARID4B,ARID4A,A375,SKCM
2,ASF1A,ASF1B,A375,SKCM
3,CCNL2,CCNL1,A375,SKCM
4,CDS1,CDS2,A375,SKCM


In [43]:
df3_new = []

for idx, row in df3.iterrows():

    df3_new.append({
        "gene1": row["gene_A"],
        "gene2": row["gene_B"],
        "cancer": row["cancer"],
        "class": 1
    })

In [44]:
df3_SL_map = {}
for g in list(set(df3["gene_A"]).union(set(df3["gene_B"]))):
    df3_SL_map[g] = []

for idx, row in df3.iterrows():
    if row["gene_B"] not in df3_SL_map[row["gene_A"]]:
        df3_SL_map[row["gene_A"]].append(row["gene_B"])
    if row["gene_A"] not in df3_SL_map[row["gene_B"]]:
        df3_SL_map[row["gene_B"]].append(row["gene_A"])

In [45]:
df3_total = pd.read_excel("../data/SL_data/raw/paralog_total.xlsx", sheet_name=0, skiprows=2)
df3_total.head()

,Pair,GENE PAIR,Gene pair class,A375_D28_rra_fdr_low,A375_D28_t_fdr_low,A375_D14_rra_fdr_low,A375_D14_t_fdr_low,Passes A375 filter?,Mewo_D28_rra_fdr_low,Mewo_D28_t_fdr_low,...,MEWO_D14_Bagel_Gene1,MEWO_D14_Bagel_Gene2,RPE_D14_Bagel_Gene1,RPE_D14_Bagel_Gene2,A375_D28_Bagel_Gene1,A375_D28_Bagel_Gene2,MEWO_D28_Bagel_Gene1,MEWO_D28_Bagel_Gene2,RPE_D28_Bagel_Gene1,RPE_D28_Bagel_Gene2
0,1,AARS2_AARS,Paralogous_gene_pair,0.228786,1.0,0.999996,1.0,No,0.999996,1.0,...,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,2,ABCB7_ABCB6,Paralogous_gene_pair,0.999996,1.0,0.999996,1.0,No,0.999996,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,ABCF1_ABCF3,Paralogous_gene_pair,0.999996,1.0,0.999996,1.0,No,0.999996,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
3,4,ABHD12B_ABHD12,Paralogous_gene_pair,0.999996,1.0,0.999996,1.0,Yes,0.999996,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,ABL1_EGFR,SynLethDB_gene_pair,0.999996,1.0,0.999996,1.0,Yes,0.999996,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [46]:

total_pairs = list(df3_total["GENE PAIR"])

for p in total_pairs:
    g1, g2 = p.split('_')[0], p.split('_')[1]
    if g1 in list(df3_SL_map.keys()):
        if g2 not in df3_SL_map[g1]:
            df3_new.append({
                "gene1": g1,
                "gene2": g2,
                "cancer": "SKCM",
                "class": 0
            })
    else:
        df3_new.append({
            "gene1": g1,
            "gene2": g2,
            "cancer": "SKCM",
            "class": 0
        })

In [47]:
df3_new_pd = pd.DataFrame(df3_new)
df3_new_pd.head()

,gene1,gene2,cancer,class
0,AP2A2,AP2A1,SKCM,1
1,ARID4B,ARID4A,SKCM,1
2,ASF1A,ASF1B,SKCM,1
3,CCNL2,CCNL1,SKCM,1
4,CDS1,CDS2,SKCM,1


In [48]:
df3_new_pd.shape

(1222, 4)

In [20]:
df3_new_pd.to_csv("../data/SL_data/processed/SLlabel_paralog_study3_binary.csv", index=False)